# Screening Credit Agentic AI — Data Preparation & EDA (v2: + RM & Current Balance)

**Capstone Project — Data Analytics**

Update dari notebook sebelumnya:
- Tabel baru **`rm_master`** (Relationship Banking Officer yang menangani tiap pengajuan)
- Kolom baru **`current_balance`** di `bank_account` (saldo real-time, terpisah dari `average_balance_6m`)

Notebook ini mencakup:
1. **Generate dataset sintetis** (8 tabel, terhubung via NIK — kecuali `rm_master` yang terhubung via `rm_id`)
2. **Join & preprocessing** — gabungkan semua tabel jadi 1 master table analitik
3. **EDA**: univariate, bivariate (terhadap label), dan korelasi antar fitur — lengkap dengan insight naratif di tiap bagian

> ⚠️ Seluruh data di notebook ini adalah **data sintetis** untuk keperluan
> pembelajaran/demo capstone — bukan data nasabah nyata.

## 0. Setup

In [1]:
!pip install -q plotly

import random
import os
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)


## 1. Generate Dataset Sintetis (8 Tabel)

Tabel yang sama seperti sebelumnya (Dukcapil, SLIK, DHN, ATR/BPN, laporan
keuangan, rekening bank), ditambah **`rm_master`** — data Relationship
Banking Officer yang menangani tiap pengajuan.

**Struktur relasi:**
- `dukcapil`, `slik_credit_history`, `dhn`, `agunan_atr_bpn`,
  `laporan_keuangan`, `bank_account` → semua terhubung ke `retail_customer_profile`
  lewat **NIK**
- `rm_master` → terhubung lewat **`rm_id`** (bukan NIK!) — karena RM itu
  pegawai bank, bukan nasabah. `retail_customer_profile` punya kolom `rm_id`
  yang jadi foreign key ke `rm_master.rm_id`
- `bank_account` sekarang juga punya kolom **`current_balance`** (saldo
  real-time) selain `average_balance_6m` (rata-rata 6 bulan) yang sudah ada
  sebelumnya

In [ ]:
# """
# Screening Credit Agentic AI - Synthetic Dataset Generator
# ============================================================
# Generate 7 tabel relasional untuk training model screening kredit retail
# banking (5C: Character, Capacity, Collateral, Condition + Capital/Identity).

# Semua tabel terhubung lewat NIK (foreign key), KECUALI retail_customer_profile
# yang punya application_id sebagai primary key (1 NIK bisa punya banyak
# application_id kalau mengajukan berkali-kali, tapi di sini kita generate
# 1 aplikasi per NIK dulu untuk versi awal).

# Cara pakai di Google Colab:
#     1. Copy semua isi file ini ke satu cell
#     2. Run
#     3. 8 file CSV akan tersimpan di /content/dataset/
#        (retail_customer_profile.csv, dukcapil.csv, slik_credit_history.csv,
#         dhn.csv, agunan_atr_bpn.csv, laporan_keuangan.csv, bank_account.csv,
#         rm_master.csv)

# PENTING saat load ulang CSV-nya nanti (termasuk di tahap join):
#     Selalu paksa NIK dibaca sebagai teks, JANGAN biarkan pandas nebak tipenya,
#     kalau tidak, 16 digit NIK bisa kepotong presisinya jadi angka:
#         pd.read_csv("dukcapil.csv", dtype={"NIK": str})

# Catatan penting: nilai tanah/bangunan per kelurahan di sini adalah ESTIMASI
# SINTETIS yang dibuat plausible per tingkatan wilayah (bukan data appraisal
# resmi/real) - cukup untuk keperluan training model & demo, BUKAN untuk
# keputusan bisnis nyata.
# """

# import random
# import numpy as np
# import pandas as pd
# from datetime import date, timedelta
# import os

# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)

# # Generator acak TERPISAH untuk field tambahan (current_balance, RM assignment)
# # supaya penambahan-penambahan ini TIDAK menggeser urutan angka acak yang
# # dipakai tabel/kolom lain yang sudah ada sebelumnya.
# CB_RNG = np.random.default_rng(SEED + 1)   # khusus current_balance
# RM_RNG = np.random.default_rng(SEED + 2)   # khusus penugasan RM

# N_CUSTOMERS = 3000          # jumlah nasabah/debitur unik
# OUT_DIR = "/content/dataset" if os.path.isdir("/content") else "./dataset"
# os.makedirs(OUT_DIR, exist_ok=True)

# # =========================================================================
# # 0. REFERENCE / LOOKUP DATA
# # =========================================================================

# FIRST_NAMES_M = ["Budi","Agus","Andi","Rizky","Dedi","Hendra","Yusuf","Fajar",
#     "Wahyu","Bambang","Eko","Rudi","Slamet","Joko","Hadi","Ahmad","Dimas",
#     "Arif","Taufik","Iwan","Gunawan","Sutrisno","Anton","Rian","Doni",
#     "Yudi","Fauzi","Irfan","Bayu","Krisna"]
# FIRST_NAMES_F = ["Siti","Dewi","Rina","Ani","Wulan","Sri","Yuni","Fitri",
#     "Indah","Lestari","Ratna","Maya","Putri","Ika","Novi","Wati","Ayu",
#     "Dian","Rita","Nina","Sari","Yanti","Lina","Desi","Tri","Retno",
#     "Kartika","Anggi","Melati","Suryani"]
# LAST_NAMES = ["Santoso","Wijaya","Kurniawan","Saputra","Setiawan","Pratama",
#     "Hidayat","Nugroho","Firmansyah","Susanto","Gunawan","Halim","Wibowo",
#     "Permana","Suryadi","Handoko","Kusuma","Rahman","Siregar","Simanjuntak",
#     "Tanjung","Lubis","Hutapea","Panjaitan","Situmorang"]

# RELIGIONS = ["ISLAM","KRISTEN","KATOLIK","HINDU","BUDDHA","KONGHUCU"]
# MARITAL = ["Menikah","Belum Menikah","Cerai Hidup","Cerai Mati"]
# EDUCATION = ["SMA/SMK","D3","S1","S2"]
# BLOOD_TYPE = ["A","B","AB","O"]

# INDUSTRIES = {
#     "Perdagangan": ["Distributor Elektronik","Toko Sembako","Grosir Pakaian",
#                     "Distributor Bahan Bangunan","Toko Alat Tulis"],
#     "Kuliner": ["Restoran","Katering","Warung Makan","Bakery"],
#     "Jasa": ["Bengkel","Laundry","Percetakan","Jasa Konstruksi Kecil"],
#     "Manufaktur": ["Konveksi","Furniture","Pengolahan Makanan Ringan"],
#     "Pertanian": ["Distributor Hasil Tani","Peternakan Ayam"],
#     "Transportasi": ["Ekspedisi Kecil","Rental Kendaraan"],
# }
# INDUSTRY_RISK = {
#     "Perdagangan": 0.05, "Kuliner": 0.10, "Jasa": 0.05,
#     "Manufaktur": 0.08, "Pertanian": 0.15, "Transportasi": 0.12,
# }

# PROVINCES_CITIES = {
#     "DKI Jakarta": ["Jakarta Selatan","Jakarta Pusat","Jakarta Timur","Jakarta Barat","Jakarta Utara"],
#     "Jawa Barat": ["Bekasi","Depok","Bogor","Tangerang Selatan"],
#     "Banten": ["Tangerang"],
# }
# REGIONS = ["Region 1","Region 2","Region 3","Region 4"]
# BRANCHES = ["KCP Tebet","KCP Kelapa Gading","KCP Bekasi Barat","KCP Depok Margonda",
#     "KCP Bogor Baranangsiang","KCP Tangerang BSD","KCP Cikini","KCP Kemang",
#     "KCP Pluit","KCP Cibubur"]

# BRANCH_TO_REGION = {
#     "KCP Tebet": "Region 1", "KCP Cikini": "Region 1", "KCP Kemang": "Region 1",
#     "KCP Pluit": "Region 2", "KCP Kelapa Gading": "Region 2", "KCP Cibubur": "Region 2",
#     "KCP Bekasi Barat": "Region 3", "KCP Depok Margonda": "Region 3",
#     "KCP Bogor Baranangsiang": "Region 4", "KCP Tangerang BSD": "Region 4",
# }
# RM_PER_BRANCH = 4
# RM_LEVELS = ["Junior RB", "Senior RB"]

# KELURAHAN_LOOKUP = [
#     ("DKI Jakarta","Jakarta Selatan","Tebet","Tebet Timur", 28, 5.5),
#     ("DKI Jakarta","Jakarta Selatan","Kebayoran Baru","Gunung",45, 6.0),
#     ("DKI Jakarta","Jakarta Selatan","Pancoran","Duren Tiga", 30, 5.5),
#     ("DKI Jakarta","Jakarta Pusat","Menteng","Menteng", 55, 6.5),
#     ("DKI Jakarta","Jakarta Pusat","Cikini","Cikini", 40, 6.0),
#     ("DKI Jakarta","Jakarta Timur","Kramat Jati","Kramat Jati", 18, 4.5),
#     ("DKI Jakarta","Jakarta Timur","Cakung","Cakung Barat", 12, 4.0),
#     ("DKI Jakarta","Jakarta Barat","Kebon Jeruk","Sukabumi Selatan", 22, 5.0),
#     ("DKI Jakarta","Jakarta Barat","Cengkareng","Cengkareng Barat", 16, 4.2),
#     ("DKI Jakarta","Jakarta Utara","Kelapa Gading","Kelapa Gading Barat", 25, 5.2),
#     ("DKI Jakarta","Jakarta Utara","Pluit","Pluit", 27, 5.3),
#     ("Jawa Barat","Bekasi","Bekasi Barat","Bintara", 9, 3.8),
#     ("Jawa Barat","Bekasi","Bekasi Timur","Margahayu", 8, 3.6),
#     ("Jawa Barat","Depok","Beji","Kemiri Muka", 10, 3.8),
#     ("Jawa Barat","Depok","Sukmajaya","Mekarjaya", 8.5, 3.6),
#     ("Jawa Barat","Bogor","Bogor Tengah","Paledang", 7, 3.4),
#     ("Jawa Barat","Tangerang Selatan","Serpong","Rawa Buntu", 12, 4.0),
#     ("Banten","Tangerang","Karawaci","Bojong Jaya", 9, 3.6),
#     ("Banten","Tangerang","Cipondoh","Poris Plawad", 7.5, 3.4),
# ]

# ASSET_TYPES = ["Tanah","Rumah","Ruko","Gudang"]
# CERT_TYPES = ["SHM","HGB"]
# LOAN_TYPES = ["KMK","KI","KPR","KKB","KK"]
# COLLECT_MAP = {1:"Lancar", 2:"Dalam Perhatian Khusus (DPK)", 3:"Kurang Lancar",
#                 4:"Diragukan", 5:"Macet"}
# OTHER_BANKS = ["Bank Mandiri","Bank BCA","Bank BRI","Bank BNI","Bank CIMB Niaga",
#     "Bank Danamon","Bank Permata","Bank OCBC NISP","Bank Panin","BPR Mitra Usaha"]
# DHN_REASONS = ["Tunggakan kredit >90 hari di bank lain","Terlibat kasus fraud dokumen",
#     "Kredit macet yang belum diselesaikan","Cek/giro kosong berulang",
#     "Laporan pihak ketiga terkait sengketa usaha"]

# def random_date(start_year, end_year):
#     start = date(start_year, 1, 1)
#     end = date(end_year, 8, 22)
#     delta = (end - start).days
#     return start + timedelta(days=random.randint(0, delta))

# KODE_WILAYAH = {
#     "Jakarta Selatan": "317401", "Jakarta Pusat": "317101", "Jakarta Timur": "317501",
#     "Jakarta Barat": "317301", "Jakarta Utara": "317201",
#     "Bekasi": "327501", "Depok": "327601", "Bogor": "327101",
#     "Tangerang Selatan": "367401", "Tangerang": "367101",
# }

# def gen_nik(kota, tanggal_lahir, gender, idx):
#     wilayah = KODE_WILAYAH.get(kota, "310101")
#     d = tanggal_lahir.day + (40 if gender == "Perempuan" else 0)
#     m, y = tanggal_lahir.month, tanggal_lahir.year % 100
#     return f"{wilayah}{d:02d}{m:02d}{y:02d}{idx:04d}"


# def generate_dukcapil(n):
#     rows = []
#     for i in range(1, n+1):
#         gender = random.choice(["Laki-Laki","Perempuan"])
#         fname = random.choice(FIRST_NAMES_M if gender=="Laki-Laki" else FIRST_NAMES_F)
#         lname = random.choice(LAST_NAMES)
#         nama = f"{fname} {lname}"
#         prov = random.choice(list(PROVINCES_CITIES.keys()))
#         kota = random.choice(PROVINCES_CITIES[prov])
#         tgl_lahir = random_date(1965, 2003)
#         nik = gen_nik(kota, tgl_lahir, gender, i)
#         rows.append({
#             "dukcapil_id": f"DKC{i:06d}",
#             "NIK": nik,
#             "nama": nama,
#             "tempat_lahir": kota,
#             "tanggal_lahir": tgl_lahir.isoformat(),
#             "jenis_kelamin": gender,
#             "golongan_darah": random.choice(BLOOD_TYPE),
#             "alamat": f"Jl. {random.choice(LAST_NAMES)} No. {random.randint(1,150)}",
#             "rt_rw": f"{random.randint(1,12):03d}/{random.randint(1,10):03d}",
#             "kelurahan_desa": random.choice(["Sukamaju","Sukajadi","Cempaka Putih",
#                 "Kebon Baru","Duren Sawit","Rawa Bunga","Cipete","Bintaro"]),
#             "kecamatan": random.choice(["Tebet","Kramat Jati","Cengkareng",
#                 "Bekasi Timur","Sukmajaya","Serpong"]),
#             "kota_kabupaten": kota,
#             "provinsi": prov,
#             "agama": random.choice(RELIGIONS),
#             "status_perkawinan": random.choice(MARITAL),
#             "pekerjaan": "Wiraswasta",
#             "kewarganegaraan": "WNI",
#             "berlaku_hingga": "SEUMUR HIDUP",
#         })
#     return pd.DataFrame(rows)


# def generate_agunan(dukcapil_df):
#     rows = []
#     agunan_lookup = {}
#     for i, r in enumerate(dukcapil_df.itertuples(), start=1):
#         nik = r.NIK
#         prov, kota, kec, kel, harga_tanah, harga_bangunan = random.choice(KELURAHAN_LOOKUP)
#         asset_type = random.choices(ASSET_TYPES, weights=[0.25,0.30,0.35,0.10])[0]
#         land_area = round(np.random.uniform(60, 400), 1)
#         building_area = 0.0 if asset_type == "Tanah" else round(land_area * np.random.uniform(0.5, 1.3), 1)
#         htn = round(harga_tanah * np.random.uniform(0.85, 1.15), 2)
#         hbg = round(harga_bangunan * np.random.uniform(0.85, 1.15), 2)
#         nilai_tanah = round(land_area * htn * 1_000_000)
#         nilai_bangunan = round(building_area * hbg * 1_000_000)
#         total_value = nilai_tanah + nilai_bangunan
#         ownership_match = np.random.choice(["Ya","Tidak"], p=[0.94, 0.06])
#         row = {
#             "atr_bpn_id": f"ATR{i:06d}",
#             "NIK": nik,
#             "asset_type": asset_type,
#             "certificate_type": random.choice(CERT_TYPES),
#             "certificate_number": f"{random.randint(10000,99999)}/{kel}",
#             "provinsi": prov, "kota": kota, "kecamatan": kec, "kelurahan": kel,
#             "land_area_m2": land_area,
#             "building_area_m2": building_area,
#             "nilai_tanah_per_m2": int(htn * 1_000_000),
#             "nilai_bangunan_per_m2": int(hbg * 1_000_000),
#             "nilai_tanah_total": nilai_tanah,
#             "nilai_bangunan_total": nilai_bangunan,
#             "total_collateral_value": total_value,
#             "ownership_match": ownership_match,
#         }
#         rows.append(row)
#         agunan_lookup[nik] = row
#     return pd.DataFrame(rows), agunan_lookup


# def generate_slik(dukcapil_df):
#     rows = []
#     slik_summary = {}
#     rid = 1
#     for r in dukcapil_df.itertuples():
#         nik = r.NIK
#         n_loans = np.random.choice([0,1,2,3], p=[0.15,0.40,0.30,0.15])
#         worst = 1
#         total_installment = 0
#         for _ in range(n_loans):
#             plafond = int(np.random.choice([25,50,75,100,150,200,300,500]) * 1_000_000)
#             outstanding = int(plafond * np.random.uniform(0.2, 0.95))
#             tenor = int(np.random.choice([12,24,36,48,60]))
#             installment = int(plafond / tenor * np.random.uniform(1.02,1.15))
#             collect = np.random.choice([1,2,3,4,5], p=[0.72,0.14,0.07,0.04,0.03])
#             worst = max(worst, collect)
#             total_installment += installment
#             rows.append({
#                 "slik_record_id": f"SLK{rid:06d}",
#                 "NIK": nik,
#                 "inquiry_date": random_date(2024,2026).isoformat(),
#                 "bank_name": random.choice(OTHER_BANKS),
#                 "loan_type": random.choice(LOAN_TYPES),
#                 "plafond": plafond,
#                 "outstanding_balance": outstanding,
#                 "installment_amount": installment,
#                 "tenor_month": tenor,
#                 "collectability": int(collect),
#                 "collectability_label": COLLECT_MAP[collect],
#             })
#             rid += 1
#         slik_summary[nik] = {"worst_collect": worst, "total_installment": total_installment, "n_loans": n_loans}
#     return pd.DataFrame(rows), slik_summary


# def generate_dhn(dukcapil_df, slik_summary):
#     rows = []
#     dhn_lookup = {}
#     for i, r in enumerate(dukcapil_df.itertuples(), start=1):
#         nik = r.NIK
#         worst = slik_summary[nik]["worst_collect"]
#         p_blacklist = {1:0.01, 2:0.03, 3:0.10, 4:0.25, 5:0.45}[worst]
#         status = np.random.choice(["Ya","Tidak"], p=[p_blacklist, 1-p_blacklist])
#         reason = random.choice(DHN_REASONS) if status == "Ya" else ""
#         row = {
#             "dhn_id": f"DHN{i:06d}",
#             "NIK": nik,
#             "status_dhn": status,
#             "alasan": reason,
#             "tanggal_input": random_date(2023,2026).isoformat(),
#         }
#         rows.append(row)
#         dhn_lookup[nik] = status
#     return pd.DataFrame(rows), dhn_lookup


# def generate_laporan_keuangan(dukcapil_df):
#     rows = []
#     fin_summary = {}
#     rid = 1
#     for r in dukcapil_df.itertuples():
#         nik = r.NIK
#         revenue_2024 = np.random.lognormal(mean=16.8, sigma=0.6)
#         growth = np.random.normal(0.12, 0.20)
#         revenue_2025 = revenue_2024 * (1 + growth)
#         margin = np.clip(np.random.normal(0.11, 0.05), 0.01, 0.35)
#         recs = []
#         for yr, rev in [(2024, revenue_2024), (2025, revenue_2025)]:
#             net_profit = rev * margin * np.random.uniform(0.85,1.15)
#             total_asset = rev * np.random.uniform(1.1, 2.0)
#             total_liability = total_asset * np.random.uniform(0.2, 0.7)
#             op_cf = net_profit * np.random.uniform(0.8, 1.4)
#             row = {
#                 "laporan_id": f"FIN{rid:06d}", "NIK": nik, "year": yr,
#                 "revenue": int(rev), "net_profit": int(net_profit),
#                 "total_asset": int(total_asset), "total_liability": int(total_liability),
#                 "operating_cashflow": int(op_cf),
#             }
#             rows.append(row); recs.append(row); rid += 1
#         fin_summary[nik] = {
#             "revenue_growth": growth,
#             "latest_revenue": recs[1]["revenue"],
#             "latest_net_profit": recs[1]["net_profit"],
#             "latest_liability": recs[1]["total_liability"],
#         }
#     return pd.DataFrame(rows), fin_summary


# def generate_bank_account(dukcapil_df, fin_summary):
#     rows = []
#     cf_summary = {}
#     aid = 1
#     for r in dukcapil_df.itertuples():
#         nik = r.NIK
#         n_acc = np.random.choice([1,2], p=[0.65,0.35])
#         monthly_rev = fin_summary[nik]["latest_revenue"] / 12
#         best_avg_balance = 0
#         for _ in range(n_acc):
#             avg_credit = monthly_rev * np.random.uniform(0.6, 1.1)
#             avg_debit = avg_credit * np.random.uniform(0.7, 0.98)
#             avg_balance = max(avg_credit - avg_debit, 0) * np.random.uniform(40, 100)
#             best_avg_balance = max(best_avg_balance, avg_balance)

#             account = {
#                 "account_id": f"ACC{aid:06d}",
#                 "NIK": nik,
#                 "account_number": f"{random.randint(1000000000,9999999999):010d}",
#                 "bank_name": random.choice(["BNI"] + OTHER_BANKS),
#                 "account_type": random.choice(["Giro","Tabungan"]),
#                 "account_status": np.random.choice(["Aktif","Dormant"], p=[0.93,0.07]),
#                 "opened_date": random_date(2015,2025).isoformat(),
#                 "average_balance_6m": int(avg_balance),
#                 "average_monthly_credit": int(avg_credit),
#                 "average_monthly_debit": int(avg_debit),
#                 "transaction_frequency_monthly": int(np.random.uniform(20,200)),
#                 "overdraft_count_6m": int(np.random.choice([0,0,0,1,2,3], p=[0.6,0.15,0.1,0.08,0.04,0.03])),
#             }

#             if account["account_status"] == "Dormant":
#                 current_balance = avg_balance * CB_RNG.uniform(0.05, 0.25)
#             elif account["overdraft_count_6m"] > 0 and CB_RNG.random() < 0.12:
#                 current_balance = -avg_debit * CB_RNG.uniform(0.02, 0.15)
#             else:
#                 current_balance = avg_balance * CB_RNG.uniform(0.4, 1.8)
#             account["current_balance"] = int(current_balance)

#             rows.append(account)
#             aid += 1
#         cf_summary[nik] = {"best_avg_balance": best_avg_balance}
#     return pd.DataFrame(rows), cf_summary


# def generate_rm_master():
#     rows = []
#     rm_lookup = {branch: [] for branch in BRANCHES}
#     rid = 1
#     for branch in BRANCHES:
#         for _ in range(RM_PER_BRANCH):
#             gender = RM_RNG.choice(["Laki-Laki", "Perempuan"])
#             fname = RM_RNG.choice(FIRST_NAMES_M if gender == "Laki-Laki" else FIRST_NAMES_F)
#             lname = RM_RNG.choice(LAST_NAMES)
#             rm_id = f"RM{rid:04d}"
#             rows.append({
#                 "rm_id": rm_id,
#                 "rm_name": f"{fname} {lname}",
#                 "branch_name": branch,
#                 "region": BRANCH_TO_REGION[branch],
#                 "jabatan": "Relationship Banking Officer",
#                 "level": RM_RNG.choice(RM_LEVELS, p=[0.6, 0.4]),
#                 "join_date": (date(2015, 1, 1) + timedelta(
#                     days=int(RM_RNG.integers(0, (date(2025, 12, 31) - date(2015, 1, 1)).days)))).isoformat(),
#             })
#             rm_lookup[branch].append(rm_id)
#             rid += 1
#     return pd.DataFrame(rows), rm_lookup


# def compute_label_score(worst_collect, dhn_status, growth, net_profit, dsr,
#                           collateral_ratio, industry):
#     s_character = {1:1.0, 2:0.8, 3:0.5, 4:0.25, 5:0.0}[worst_collect]
#     if dhn_status == "Ya":
#         s_character = min(s_character, 0.1)
#     s_capacity = np.clip(0.5 + growth*1.2, 0, 1) * 0.5 + np.clip(1 - dsr, 0, 1) * 0.5
#     s_capacity = np.clip(s_capacity, 0, 1)
#     s_collateral = np.clip(collateral_ratio / 1.5, 0, 1)
#     s_condition = 1 - INDUSTRY_RISK.get(industry, 0.1) * 4
#     s_condition = np.clip(s_condition, 0, 1)

#     score = 0.35*s_character + 0.30*s_capacity + 0.20*s_collateral + 0.15*s_condition
#     score += np.random.normal(0, 0.05)
#     return np.clip(score, 0, 1)

# def generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary, dhn_lookup,
#                                 fin_summary, cf_summary, rm_lookup):
#     rows = []
#     for i, r in enumerate(dukcapil_df.itertuples(), start=1):
#         nik = r.NIK
#         prov, kota = r.provinsi, r.kota_kabupaten
#         legal_entity = random.choice(["PT","CV","UD"])
#         industry = random.choice(list(INDUSTRIES.keys()))
#         sub_industry = random.choice(INDUSTRIES[industry])
#         business_age = int(np.random.uniform(1, 20))
#         employee_count = int(np.random.uniform(2, 80))
#         monthly_turnover = fin_summary[nik]["latest_revenue"] / 12

#         agunan = agunan_lookup[nik]
#         loan_requested = int(np.random.choice([50,75,100,150,200,300,500,750,1000]) * 1_000_000)
#         loan_requested = min(loan_requested, 10_000_000_000)
#         collateral_ratio = round(agunan["total_collateral_value"] / max(loan_requested,1), 2)
#         collateral_size_m2 = round(agunan["land_area_m2"] + agunan["building_area_m2"], 1)

#         slik = slik_summary[nik]
#         new_installment = loan_requested/36 * 1.12
#         dsr = (slik["total_installment"] + new_installment) / max(monthly_turnover, 1)

#         score = compute_label_score(
#             worst_collect=slik["worst_collect"], dhn_status=dhn_lookup[nik],
#             growth=fin_summary[nik]["revenue_growth"], net_profit=fin_summary[nik]["latest_net_profit"],
#             dsr=dsr, collateral_ratio=collateral_ratio, industry=industry,
#         )
#         label = "Diterima" if score >= 0.55 else "Ditolak"

#         row = {
#             "application_id": f"APP{2026}{i:05d}",
#             "NIK": nik,
#             "cif_number": f"CIF{1000000+i}",
#             "application_date": random_date(2025,2026).isoformat(),
#             "customer_type": "UMKM",
#             "company_name": f"{random.choice(['PT','CV','UD'])} {random.choice(LAST_NAMES)} {random.choice(['Jaya','Makmur','Sejahtera','Abadi','Mandiri'])}",
#             "legal_entity": legal_entity,
#             "owner_name": r.nama,
#             "owner_gender": "L" if r.jenis_kelamin=="Laki-Laki" else "P",
#             "owner_age": date.today().year - int(r.tanggal_lahir[:4]),
#             "owner_marital_status": r.status_perkawinan,
#             "owner_education": random.choice(EDUCATION),
#             "province": prov, "city": kota,
#             "district": r.kecamatan, "region": random.choice(REGIONS),
#             "branch_name": random.choice(BRANCHES),
#             "industry": industry, "sub_industry": sub_industry,
#             "business_age_year": business_age,
#             "employee_count": employee_count,
#             "monthly_turnover_est": int(monthly_turnover),
#             "transaction_frequency_monthly": int(np.random.uniform(30,200)),
#             "loan_requested": loan_requested,
#             "collateral_type": agunan["asset_type"],
#             "collateral_location": f"{agunan['kelurahan']}, {agunan['kota']}",
#             "collateral_province": agunan["provinsi"], "collateral_city": agunan["kota"],
#             "collateral_size_m2": collateral_size_m2,
#             "collateral_market_value": agunan["total_collateral_value"],
#             "collateral_liquidation_value": int(agunan["total_collateral_value"] * 0.8),
#             "collateral_ratio": collateral_ratio,
#             "certificate_type": agunan["certificate_type"],
#             "ownership_match": agunan["ownership_match"],
#             "estimated_dsr": round(min(dsr,3.0), 2),
#             "eligibility_score": round(float(score), 3),
#             "label": label,
#         }

#         row["rm_id"] = RM_RNG.choice(rm_lookup[row["branch_name"]])

#         rows.append(row)
#     return pd.DataFrame(rows)

In [ ]:
# print(f"Generating {N_CUSTOMERS} customers...")
# dukcapil_df = generate_dukcapil(N_CUSTOMERS)
# agunan_df, agunan_lookup = generate_agunan(dukcapil_df)
# slik_df, slik_summary = generate_slik(dukcapil_df)
# dhn_df, dhn_lookup = generate_dhn(dukcapil_df, slik_summary)
# fin_df, fin_summary = generate_laporan_keuangan(dukcapil_df)
# bank_df, cf_summary = generate_bank_account(dukcapil_df, fin_summary)
# rm_df, rm_lookup = generate_rm_master()
# profile_df = generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary,
#                                         dhn_lookup, fin_summary, cf_summary, rm_lookup)

# tables = {
#     "retail_customer_profile": profile_df, "dukcapil": dukcapil_df,
#     "slik_credit_history": slik_df, "dhn": dhn_df,
#     "agunan_atr_bpn": agunan_df, "laporan_keuangan": fin_df,
#     "bank_account": bank_df, "rm_master": rm_df,
# }
# for name, df in tables.items():
#     if "NIK" in df.columns:
#         df["NIK"] = df["NIK"].astype(str)
#     if "account_number" in df.columns:
#         df["account_number"] = df["account_number"].astype(str)
#     df.to_csv(os.path.join(OUT_DIR, f"{name}.csv"), index=False)

# print("Baris per tabel:")
# for name, df in tables.items():
#     print(f"  {name:28s} -> {len(df):6d} baris")

# print("\nDistribusi label (retail_customer_profile):")
# print(profile_df["label"].value_counts(normalize=True).round(3))


Generating 3000 customers...
Baris per tabel:
  retail_customer_profile      ->   3000 baris
  dukcapil                     ->   3000 baris
  slik_credit_history          ->   4416 baris
  dhn                          ->   3000 baris
  agunan_atr_bpn               ->   3000 baris
  laporan_keuangan             ->   6000 baris
  bank_account                 ->   4052 baris
  rm_master                    ->     40 baris

Distribusi label (retail_customer_profile):
label
Diterima    0.848
Ditolak     0.152
Name: proportion, dtype: float64


In [2]:
import os
import pandas as pd
from IPython.display import display

folder = "../data/raw"

for file in sorted(os.listdir(folder)):
    path = os.path.join(folder, file)

    if file.endswith(".csv"):
        print(f"\n{'='*60}")
        print(f"FILE: {file}")

        df = pd.read_csv(path)
        print(f"Shape: {df.shape}")
        display(df)  # menampilkan seluruh DataFrame (sesuai batas output Jupyter)


FILE: agunan_atr_bpn.csv
Shape: (3000, 17)


,atr_bpn_id,NIK,asset_type,certificate_type,certificate_number,provinsi,kota,kecamatan,kelurahan,land_area_m2,building_area_m2,nilai_tanah_per_m2,nilai_bangunan_per_m2,nilai_tanah_total,nilai_bangunan_total,total_collateral_value,ownership_match
0,ATR000001,3276010601750001,Rumah,HGB,14452/Poris Plawad,Banten,Tangerang,Cipondoh,Poris Plawad,187.3,236.1,8020000,3500000,1502146000,826350000,2328496000,Ya
1,ATR000002,3172010301920002,Rumah,HGB,36375/Pluit,DKI Jakarta,Jakarta Utara,Pluit,Pluit,113.0,61.8,29970000,5460000,3386610000,337428000,3724038000,Ya
2,ATR000003,3671010604800003,Tanah,HGB,88248/Sukabumi Selatan,DKI Jakarta,Jakarta Barat,Kebon Jeruk,Sukabumi Selatan,67.0,0.0,25100000,5500000,1681700000,0,1681700000,Ya
3,ATR000004,3173016001890004,Ruko,HGB,45461/Tebet Timur,DKI Jakarta,Jakarta Selatan,Tebet,Tebet Timur,121.8,78.8,26360000,5540000,3210648000,436552000,3647200000,Ya
4,ATR000005,3275011505030005,Ruko,HGB,41599/Kemiri Muka,Jawa Barat,Depok,Beji,Kemiri Muka,159.0,157.3,8920000,3560000,1418280000,559988000,1978268000,Ya
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,ATR002996,3671011312982996,Ruko,HGB,25511/Bintara,Jawa Barat,Bekasi,Bekasi Barat,Bintara,273.3,228.5,7820000,3790000,2137206000,866015000,3003221000,Ya
2996,ATR002997,3172010710992997,Rumah,SHM,83870/Duren Tiga,DKI Jakarta,Jakarta Selatan,Pancoran,Duren Tiga,93.6,52.0,26880000,5940000,2515968000,308880000,2824848000,Ya
2997,ATR002998,3174012706032998,Tanah,SHM,51642/Cikini,DKI Jakarta,Jakarta Pusat,Cikini,Cikini,252.1,0.0,39620000,6700000,9988202000,0,9988202000,Ya
2998,ATR002999,3276015311842999,Ruko,SHM,91026/Duren Tiga,DKI Jakarta,Jakarta Selatan,Pancoran,Duren Tiga,336.1,386.7,31280000,5430000,10513208000,2099781000,12612989000,Ya



FILE: bank_account.csv
Shape: (4052, 13)


,account_id,NIK,account_number,bank_name,account_type,account_status,opened_date,average_balance_6m,average_monthly_credit,average_monthly_debit,transaction_frequency_monthly,overdraft_count_6m,current_balance
0,ACC000001,3276010601750001,6956650690,Bank BCA,Giro,Aktif,2023-08-14,10342365,2196060,2092204,148,0,13581790
1,ACC000002,3276010601750001,6225587025,BNI,Giro,Aktif,2020-05-20,19800614,2212722,1737140,76,0,9133735
2,ACC000003,3172010301920002,4729122289,Bank BNI,Tabungan,Aktif,2025-03-21,4014256,1124770,1040039,122,0,1718267
3,ACC000004,3671010604800003,4611355005,Bank BCA,Giro,Dormant,2021-10-23,14489012,1394768,1222864,22,1,3156322
4,ACC000005,3173016001890004,8727347064,Bank CIMB Niaga,Giro,Dormant,2025-08-11,1895269,1172430,1132201,157,0,317322
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4047,ACC004048,3172010710992997,8803595257,Bank Mandiri,Giro,Aktif,2019-09-25,29578651,1228855,871975,67,0,25209840
4048,ACC004049,3172010710992997,1630495154,Bank CIMB Niaga,Giro,Aktif,2019-06-03,24206796,1709526,1369472,57,0,29703240
4049,ACC004050,3174012706032998,3421285182,Bank Mandiri,Giro,Aktif,2022-07-13,5343417,671639,544151,68,0,4880887
4050,ACC004051,3276015311842999,8950692384,Bank BNI,Tabungan,Aktif,2022-07-22,30523163,2003105,1433503,50,0,21018462



FILE: dhn.csv
Shape: (3000, 5)


...

,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,owner_marital_status,owner_education,province,city,district,region,branch_name,industry,sub_industry,business_age_year,employee_count,monthly_turnover_est,transaction_frequency_monthly,loan_requested,collateral_type,collateral_location,collateral_province,collateral_city,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,rm_id
0,APP202600001,3276010601750001,CIF1000001,2025-07-08,UMKM,UD Santoso Abadi,UD,Budi Panjaitan,L,51,Menikah,S2,Jawa Barat,Depok,Sukmajaya,Region 2,KCP Bogor Baranangsiang,Manufaktur,Konveksi,14,9,2672137,66,300000000,Rumah,"Poris Plawad, Tangerang",Banten,Tangerang,423.4,2328496000,1862796800,7.76,HGB,Ya,3.0,0.804,Diterima,RM0020
1,APP202600002,3172010301920002,CIF1000002,2025-09-01,UMKM,UD Wijaya Mandiri,UD,Andi Hidayat,L,34,Cerai Hidup,S2,DKI Jakarta,Jakarta Utara,Kramat Jati,Region 1,KCP Bekasi Barat,Jasa,Bengkel,1,3,1807917,39,75000000,Rumah,"Pluit, Jakarta Utara",DKI Jakarta,Jakarta Utara,174.8,3724038000,2979230400,49.65,HGB,Ya,3.0,0.683,Diterima,RM0010
2,APP202600003,3671010604800003,CIF1000003,2025-10-27,UMKM,UD Kusuma Sejahtera,CV,Doni Pratama,L,46,Cerai Hidup,S2,Banten,Tangerang,Bekasi Timur,Region 3,KCP Cibubur,Transportasi,Ekspedisi Kecil,17,33,1698736,59,150000000,Tanah,"Sukabumi Selatan, Jakarta Barat",DKI Jakarta,Jakarta Barat,67.0,1681700000,1345360000,11.21,HGB,Ya,3.0,0.435,Ditolak,RM0039
3,APP202600004,3173016001890004,CIF1000004,2025-10-23,UMKM,UD Susanto Makmur,CV,Nina Firmansyah,P,37,Menikah,S2,DKI Jakarta,Jakarta Barat,Sukmajaya,Region 3,KCP Bogor Baranangsiang,Perdagangan,Toko Alat Tulis,12,79,1135105,98,200000000,Ruko,"Tebet Timur, Jakarta Selatan",DKI Jakarta,Jakarta Selatan,200.6,3647200000,2917760000,18.24,HGB,Ya,3.0,0.683,Diterima,RM0017
4,APP202600005,3275011505030005,CIF1000005,2025-08-17,UMKM,CV Wijaya Sejahtera,UD,Sutrisno Nugroho,L,23,Cerai Hidup,S2,Jawa Barat,Bekasi,Kramat Jati,Region 3,KCP Bogor Baranangsiang,Transportasi,Rental Kendaraan,11,5,862772,120,750000000,Ruko,"Kemiri Muka, Depok",Jawa Barat,Depok,316.3,1978268000,1582614400,2.64,HGB,Ya,3.0,0.664,Diterima,RM0019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,APP202602996,3671011312982996,CIF1002996,2026-01-20,UMKM,PT Nugroho Makmur,UD,Yusuf Siregar,L,28,Cerai Mati,S2,Banten,Tangerang,Tebet,Region 4,KCP Bekasi Barat,Manufaktur,Pengolahan Makanan Ringan,4,48,4888195,93,750000000,Ruko,"Bintara, Bekasi",Jawa Barat,Bekasi,501.8,3003221000,2402576800,4.00,HGB,Ya,3.0,0.733,Diterima,RM0009
2996,APP202602997,3172010710992997,CIF1002997,2026-01-11,UMKM,CV Setiawan Makmur,PT,Joko Wibowo,L,27,Cerai Mati,S2,DKI Jakarta,Jakarta Utara,Sukmajaya,Region 3,KCP Kelapa Gading,Jasa,Laundry,5,30,1709930,127,300000000,Rumah,"Duren Tiga, Jakarta Selatan",DKI Jakarta,Jakarta Selatan,145.6,2824848000,2259878400,9.42,SHM,Ya,3.0,0.717,Diterima,RM0006
2997,APP202602998,3174012706032998,CIF1002998,2025-05-25,UMKM,UD Wijaya Abadi,CV,Budi Lubis,L,23,Menikah,SMA/SMK,DKI Jakarta,Jakarta Selatan,Tebet,Region 3,KCP Cibubur,Kuliner,Bakery,8,69,719843,166,50000000,Tanah,"Cikini, Jakarta Pusat",DKI Jakarta,Jakarta Pusat,252.1,9988202000,7990561600,199.76,SHM,Ya,3.0,0.640,Diterima,RM0039
2998,APP202602999,3276015311842999,CIF1002999,2026-07-28,UMKM,CV Permana Jaya,UD,Putri Handoko,P,42,Belum Menikah,D3,Jawa Barat,Depok,Cengkareng,Region 3,KCP Tangerang BSD,Perdagangan,Distributor Bahan Bangunan,11,31,2304116,138,750000000,Ruko,"Duren Tiga, Jakarta Selatan",DKI Jakarta,Jakarta Selatan,722.8,12612989000,10090391200,16.82,SHM,Ya,3.0,0.654,Diterima,RM0023



FILE: rm_master.csv
Shape: (40, 7)


,rm_id,rm_name,branch_name,region,jabatan,level,join_date
0,RM0001,Ani Tanjung,KCP Tebet,Region 1,Relationship Banking Officer,Junior RB,2017-11-02
1,RM0002,Krisna Permana,KCP Tebet,Region 1,Relationship Banking Officer,Senior RB,2016-10-14
2,RM0003,Dedi Kurniawan,KCP Tebet,Region 1,Relationship Banking Officer,Senior RB,2018-09-19
3,RM0004,Rita Halim,KCP Tebet,Region 1,Relationship Banking Officer,Junior RB,2025-07-02
4,RM0005,Bayu Situmorang,KCP Kelapa Gading,Region 2,Relationship Banking Officer,Senior RB,2025-03-11
5,RM0006,Budi Suryadi,KCP Kelapa Gading,Region 2,Relationship Banking Officer,Senior RB,2023-10-09
6,RM0007,Wati Lubis,KCP Kelapa Gading,Region 2,Relationship Banking Officer,Junior RB,2017-03-03
7,RM0008,Retno Kusuma,KCP Kelapa Gading,Region 2,Relationship Banking Officer,Junior RB,2015-12-17
8,RM0009,Yuni Pratama,KCP Bekasi Barat,Region 3,Relationship Banking Officer,Senior RB,2018-04-25
9,RM0010,Indah Rahman,KCP Bekasi Barat,Region 3,Relationship Banking Officer,Senior RB,2022-07-08



FILE: slik_credit_history.csv
Shape: (4416, 11)


,slik_record_id,NIK,inquiry_date,bank_name,loan_type,plafond,outstanding_balance,installment_amount,tenor_month,collectability,collectability_label
0,SLK000001,3276010601750001,2024-03-06,Bank BCA,KPR,100000000,87078357,2882804,36,1,Lancar
1,SLK000002,3172010301920002,2025-03-27,Bank CIMB Niaga,KKB,75000000,42479157,6708701,12,2,Dalam Perhatian Khusus (DPK)
2,SLK000003,3172010301920002,2024-06-29,BPR Mitra Usaha,KK,300000000,176591612,9447252,36,1,Lancar
3,SLK000004,3671010604800003,2024-02-26,Bank Danamon,KPR,25000000,5032340,760302,36,1,Lancar
4,SLK000005,3671010604800003,2024-11-25,Bank BRI,KI,150000000,130655336,14335826,12,4,Diragukan
...,...,...,...,...,...,...,...,...,...,...,...
4411,SLK004412,3174012706032998,2024-09-28,Bank OCBC NISP,KKB,500000000,308828799,14985195,36,1,Lancar
4412,SLK004413,3174012706032998,2026-01-24,Bank BRI,KMK,150000000,126738850,3523020,48,1,Lancar
4413,SLK004414,3275016008853000,2026-02-25,Bank CIMB Niaga,KPR,100000000,41090820,8924469,12,1,Lancar
4414,SLK004415,3275016008853000,2024-08-18,Bank BNI,KKB,150000000,125339004,4760918,36,1,Lancar


## 2. Join & Preprocessing

Tabel `slik_credit_history`, `bank_account`, `laporan_keuangan` tetap
**1:banyak** terhadap NIK → diagregasi dulu sebelum join (`current_balance`
sekarang ikut diagregasi juga: `bank_current_balance_total` dan
`bank_best_current_balance`).

**`rm_master` beda pola join-nya**: karena relasinya **1:1 terhadap
`rm_id`** (1 pengajuan = tepat 1 RM, dan `rm_master` juga 1 baris = 1 RM),
cukup **LEFT JOIN biasa lewat `rm_id`** — TIDAK perlu diagregasi seperti
tabel 1:banyak lainnya.

Preprocessing lain yang dilakukan:
- Isi NaN hasil agregasi SLIK dengan nilai bermakna (0 = belum ada riwayat)
- Isi `dhn_alasan` kosong dengan "Tidak Berlaku"
- Kolom yang **HARUS di-drop sebelum modeling**, dibagi 3 kategori:
  - **Leakage**: `eligibility_score` (sumber langsung rumus label)
  - **Sensitif (fair-lending)**: `owner_gender`, `owner_marital_status`
  - **Operasional RM (governance)**: `rm_id`, `rm_name`, `rm_branch_name`,
    dll — kalau dipakai jadi fitur model, berisiko mengunci bias/favoritism
    RM individual ke dalam sistem scoring, bukan menilai kelayakan nasabah
    murni. Kolom ini tetap berguna untuk **monitoring** (bukan scoring).

In [4]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

NIK_STR = {"NIK": str}

profile  = pd.read_csv(f"{RAW_DIR}/retail_customer_profile.csv", dtype=NIK_STR)
dukcapil = pd.read_csv(f"{RAW_DIR}/dukcapil.csv", dtype=NIK_STR)
slik     = pd.read_csv(f"{RAW_DIR}/slik_credit_history.csv", dtype=NIK_STR)
dhn      = pd.read_csv(f"{RAW_DIR}/dhn.csv", dtype=NIK_STR)
agunan   = pd.read_csv(f"{RAW_DIR}/agunan_atr_bpn.csv", dtype=NIK_STR)
fin      = pd.read_csv(f"{RAW_DIR}/laporan_keuangan.csv", dtype=NIK_STR)
bank     = pd.read_csv(f"{RAW_DIR}/bank_account.csv", dtype={**NIK_STR, "account_number": str})
rm       = pd.read_csv(f"{RAW_DIR}/rm_master.csv")

# --- Agregasi tabel 1:banyak terhadap NIK ---
slik_agg = (slik.groupby("NIK")
    .agg(slik_n_loans=("slik_record_id","count"),
         slik_worst_collectability=("collectability","max"),
         slik_n_banks=("bank_name","nunique"),
         slik_total_outstanding=("outstanding_balance","sum"),
         slik_total_installment_other=("installment_amount","sum"),
         slik_avg_tenor_month=("tenor_month","mean"))
    .reset_index())
slik_agg["slik_has_macet"] = (slik_agg["slik_worst_collectability"] == 5).astype(int)
slik_agg["slik_has_credit_history"] = 1

# bank_account: sekarang termasuk agregasi current_balance juga
bank_agg = (bank.groupby("NIK")
    .agg(bank_n_accounts=("account_id","count"),
         bank_best_avg_balance_6m=("average_balance_6m","max"),
         bank_total_avg_credit=("average_monthly_credit","sum"),
         bank_total_avg_debit=("average_monthly_debit","sum"),
         bank_total_overdraft_6m=("overdraft_count_6m","sum"),
         bank_current_balance_total=("current_balance","sum"),
         bank_best_current_balance=("current_balance","max"))
    .reset_index())
bank_agg["bank_any_dormant"] = bank.groupby("NIK")["account_status"] \
    .apply(lambda s: int((s == "Dormant").any())).values

fin_pivot = fin.pivot_table(index="NIK", columns="year",
    values=["revenue","net_profit","total_asset","total_liability","operating_cashflow"])
fin_pivot.columns = [f"{col}_{yr}" for col, yr in fin_pivot.columns]
fin_pivot = fin_pivot.reset_index()
fin_pivot["revenue_growth_pct"] = ((fin_pivot["revenue_2025"] - fin_pivot["revenue_2024"])
                                     / fin_pivot["revenue_2024"]).round(4)
fin_pivot["profit_margin_2025"] = (fin_pivot["net_profit_2025"] / fin_pivot["revenue_2025"]).round(4)
fin_pivot["liability_to_asset_2025"] = (fin_pivot["total_liability_2025"]
                                          / fin_pivot["total_asset_2025"]).round(4)

agunan_extra = agunan[["NIK","kelurahan","land_area_m2","building_area_m2",
                        "nilai_tanah_per_m2","nilai_bangunan_per_m2"]].rename(
    columns={"kelurahan":"agunan_kelurahan"})
dhn_slim = dhn[["NIK","status_dhn","alasan"]].rename(columns={"alasan":"dhn_alasan"})

# rm_master: relasinya 1:1 terhadap rm_id (BUKAN NIK). retail_customer_profile
# sudah punya kolom rm_id (1 pengajuan = tepat 1 RM), jadi cukup LEFT JOIN
# biasa lewat rm_id - TIDAK perlu diagregasi seperti slik/bank_account/
# laporan_keuangan (yang relasinya 1:banyak terhadap NIK).
rm_slim = rm.rename(columns={"branch_name": "rm_branch_name", "region": "rm_region"})

# --- Join semuanya ke master table ---
master = (profile
    .merge(dhn_slim, on="NIK", how="left")
    .merge(slik_agg, on="NIK", how="left")
    .merge(bank_agg, on="NIK", how="left")
    .merge(fin_pivot, on="NIK", how="left")
    .merge(agunan_extra, on="NIK", how="left")
    .merge(rm_slim, on="rm_id", how="left"))

# --- Preprocessing ---
slik_num_cols = ["slik_n_loans","slik_worst_collectability","slik_n_banks",
                  "slik_total_outstanding","slik_total_installment_other",
                  "slik_avg_tenor_month","slik_has_macet"]
master[slik_num_cols] = master[slik_num_cols].fillna(0)
master["slik_has_credit_history"] = master["slik_has_credit_history"].fillna(0).astype(int)
master["dhn_alasan"] = master["dhn_alasan"].fillna("Tidak Berlaku")
master["application_date"] = pd.to_datetime(master["application_date"])

master["dsr_capped"] = master["estimated_dsr"].clip(upper=3.0)
master["is_female_owner"] = (master["owner_gender"] == "P").astype(int)
master["has_dhn_flag"] = (master["status_dhn"] == "Ya").astype(int)

missing = master.isna().sum()
missing = missing[missing > 0]
print("Kolom yang masih missing setelah preprocessing:", list(missing.index) if len(missing) else "(tidak ada)")

LEAKAGE_COLS = ["eligibility_score"]
SENSITIVE_COLS = ["owner_gender", "is_female_owner", "owner_marital_status"]
OPERATIONAL_ONLY_COLS = ["rm_id", "rm_name", "rm_branch_name", "rm_region", "jabatan", "level", "join_date"]
print(f"Kolom leakage (WAJIB drop sblm modeling): {LEAKAGE_COLS}")
print(f"Kolom sensitif (drop dari fitur model, fair-lending): {SENSITIVE_COLS}")
print(f"Kolom operasional RM (drop dari fitur model, governance): {OPERATIONAL_ONLY_COLS}")

master.to_csv(f"{PROCESSED_DIR}/master_dataset.csv", index=False)
print(f"\nMaster table: {master.shape[0]} baris x {master.shape[1]} kolom")
master.head(3)


Kolom yang masih missing setelah preprocessing: (tidak ada)
Kolom leakage (WAJIB drop sblm modeling): ['eligibility_score']
Kolom sensitif (drop dari fitur model, fair-lending): ['owner_gender', 'is_female_owner', 'owner_marital_status']
Kolom operasional RM (drop dari fitur model, governance): ['rm_id', 'rm_name', 'rm_branch_name', 'rm_region', 'jabatan', 'level', 'join_date']

Master table: 3000 baris x 83 kolom


,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,owner_marital_status,owner_education,province,city,district,region,branch_name,industry,sub_industry,business_age_year,employee_count,monthly_turnover_est,transaction_frequency_monthly,loan_requested,collateral_type,collateral_location,collateral_province,collateral_city,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,rm_id,status_dhn,dhn_alasan,slik_n_loans,slik_worst_collectability,slik_n_banks,slik_total_outstanding,slik_total_installment_other,slik_avg_tenor_month,slik_has_macet,slik_has_credit_history,bank_n_accounts,bank_best_avg_balance_6m,bank_total_avg_credit,bank_total_avg_debit,bank_total_overdraft_6m,bank_current_balance_total,bank_best_current_balance,bank_any_dormant,net_profit_2024,net_profit_2025,operating_cashflow_2024,operating_cashflow_2025,revenue_2024,revenue_2025,total_asset_2024,total_asset_2025,total_liability_2024,total_liability_2025,revenue_growth_pct,profit_margin_2025,liability_to_asset_2025,agunan_kelurahan,land_area_m2,building_area_m2,nilai_tanah_per_m2,nilai_bangunan_per_m2,rm_name,rm_branch_name,rm_region,jabatan,level,join_date,dsr_capped,is_female_owner,has_dhn_flag
0,APP202600001,3276010601750001,CIF1000001,2025-07-08,UMKM,UD Santoso Abadi,UD,Budi Panjaitan,L,51,Menikah,S2,Jawa Barat,Depok,Sukmajaya,Region 2,KCP Bogor Baranangsiang,Manufaktur,Konveksi,14,9,2672137,66,300000000,Rumah,"Poris Plawad, Tangerang",Banten,Tangerang,423.4,2328496000,1862796800,7.76,HGB,Ya,3.0,0.804,Diterima,RM0020,Tidak,Tidak Berlaku,1.0,1.0,1.0,87078357.0,2882804.0,36.0,0.0,1,2,19800614,4408782,3829344,0,22715525,13581790,0,3405584.0,3318494.0,3039455.0,4499938.0,29666491.0,32065651.0,48148218.0,37084511.0,27015813.0,18787312.0,0.0809,0.1035,0.5066,Poris Plawad,187.3,236.1,8020000,3500000,Indah Kusuma,KCP Bogor Baranangsiang,Region 4,Relationship Banking Officer,Junior RB,2017-04-09,3.0,0,0
1,APP202600002,3172010301920002,CIF1000002,2025-09-01,UMKM,UD Wijaya Mandiri,UD,Andi Hidayat,L,34,Cerai Hidup,S2,DKI Jakarta,Jakarta Utara,Kramat Jati,Region 1,KCP Bekasi Barat,Jasa,Bengkel,1,3,1807917,39,75000000,Rumah,"Pluit, Jakarta Utara",DKI Jakarta,Jakarta Utara,174.8,3724038000,2979230400,49.65,HGB,Ya,3.0,0.683,Diterima,RM0010,Tidak,Tidak Berlaku,2.0,2.0,2.0,219070769.0,16155953.0,24.0,0.0,1,1,4014256,1124770,1040039,0,1718267,1718267,0,2907631.0,2602253.0,3572130.0,2863796.0,22347316.0,21695009.0,27256315.0,35323008.0,17484128.0,9380894.0,-0.0292,0.1199,0.2656,Pluit,113.0,61.8,29970000,5460000,Indah Rahman,KCP Bekasi Barat,Region 3,Relationship Banking Officer,Senior RB,2022-07-08,3.0,0,0
2,APP202600003,3671010604800003,CIF1000003,2025-10-27,UMKM,UD Kusuma Sejahtera,CV,Doni Pratama,L,46,Cerai Hidup,S2,Banten,Tangerang,Bekasi Timur,Region 3,KCP Cibubur,Transportasi,Ekspedisi Kecil,17,33,1698736,59,150000000,Tanah,"Sukabumi Selatan, Jakarta Barat",DKI Jakarta,Jakarta Barat,67.0,1681700000,1345360000,11.21,HGB,Ya,3.0,0.435,Ditolak,RM0039,Tidak,Tidak Berlaku,2.0,4.0,2.0,135687676.0,15096128.0,24.0,0.0,1,1,14489012,1394768,1222864,1,3156322,3156322,1,5615079.0,3670504.0,7404291.0,4126254.0,23914503.0,20384842.0,34076107.0,37104928.0,17322819.0,11862908.0,-0.1476,0.1801,0.3197,Sukabumi Selatan,67.0,0.0,25100000,5500000,Slamet Hutapea,KCP Cibubur,Region 2,Relationship Banking Officer,Senior RB,2021-10-25,3.0,0,0


## 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate — Distribusi Tiap Fitur Penting

**3.1.1 Distribusi Label (Target)**

In [5]:
label_counts = master["label"].value_counts().reset_index()
label_counts.columns = ["label", "jumlah"]

fig = px.pie(label_counts, names="label", values="jumlah", hole=0.45,
             color="label", color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Distribusi Label (Target)")
fig.update_traces(textinfo="percent+label")
fig.show()


**Insight:** Dari 3.000 pengajuan, **84,8% Diterima** dan **15,2%
Ditolak**. Distribusi ini cukup imbalanced — kalau nanti dipakai untuk
training model klasifikasi, perlu teknik penanganan imbalance (class
weighting, oversampling minority class, atau evaluasi pakai metrik yang
tidak bias ke kelas mayoritas seperti F1-score/AUC, bukan cuma accuracy).

**3.1.2 Kolektibilitas SLIK Terburuk**

In [6]:
collect_label_map = {0: "Belum Ada Riwayat", 1: "Lancar", 2: "DPK",
                      3: "Kurang Lancar", 4: "Diragukan", 5: "Macet"}
order = ["Belum Ada Riwayat","Lancar","DPK","Kurang Lancar","Diragukan","Macet"]
vc = master["slik_worst_collectability"].map(collect_label_map).value_counts().reindex(order).reset_index()
vc.columns = ["kolektibilitas", "jumlah"]

fig = px.bar(vc, x="kolektibilitas", y="jumlah",
             title="Distribusi Kolektibilitas SLIK Terburuk",
             labels={"kolektibilitas": "Kolektibilitas", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** Mayoritas (~50%) nasabah Lancar, ~14% belum ada riwayat
kredit (nasabah baru — perlu dinilai netral, bukan disamakan dgn Lancar
maupun bermasalah), dan ~4,3% sudah Macet. Proporsi ini realistis untuk
portofolio kredit UMKM retail.

**3.1.3 Status Daftar Hitam Nasional (DHN)**

In [7]:
vc = master["status_dhn"].value_counts().reset_index()
vc.columns = ["status_dhn", "jumlah"]

fig = px.bar(vc, x="status_dhn", y="jumlah", color="status_dhn",
             color_discrete_map={"Tidak": "#2ecc71", "Ya": "#e74c3c"},
             title="Distribusi Status Daftar Hitam Nasional (DHN)",
             labels={"status_dhn": "Status DHN", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** 4,9% nasabah (147 dari 3.000) terdaftar di DHN. Proporsi
ini didesain berkorelasi dengan kolektibilitas SLIK — makin buruk riwayat
kredit, makin besar peluang masuk DHN (lihat bagian korelasi/bivariate).

**3.1.4 Estimasi DSR (Debt Service Ratio)**

In [8]:
fig = px.histogram(master, x="estimated_dsr", nbins=40,
                    title="Distribusi Estimasi DSR (Debt Service Ratio)",
                    labels={"estimated_dsr": "Estimasi DSR"})
fig.add_vline(x=3.0, line_dash="dash", line_color="red",
              annotation_text="batas cap (3.0)")
fig.show()


**Insight — perlu perhatian khusus:** Nilai `estimated_dsr` sangat
menumpuk di batas cap 3,0 (Q1 dan median-nya SAMA-SAMA sudah di titik cap).
Artinya mayoritas nasabah punya cicilan yang jauh melebihi kapasitas bayar
bulanan mereka secara estimasi kasar — fitur ini **kurang diskriminatif**
dalam bentuk sekarang karena variasinya kecil (banyak nasabah "terlihat
sama" di titik maksimum). Kalau lanjut ke modeling, pertimbangkan
transformasi (log, atau naikkan batas cap) supaya sinyalnya lebih tajam.

**3.1.5 Collateral Ratio**

In [9]:
fig = px.histogram(master, x="collateral_ratio", nbins=60, log_y=True,
                    title="Distribusi Collateral Ratio (sumbu-y log scale)",
                    labels={"collateral_ratio": "Collateral Ratio (nilai agunan / pinjaman)"})
fig.show()


**Insight:** Distribusi sangat right-skewed (makanya sumbu-y dipakai log
scale) — banyak nasabah over-collateralized jauh di atas nilai pinjaman
yang diajukan. Ini karena nilai agunan (dari lookup harga tanah/bangunan
per kelurahan) di-generate independen dari nominal pinjaman, jadi banyak
kombinasi "kebetulan" agunan besar + pinjaman kecil. Perlu dicatat sebagai
limitation dataset sintetis di laporan.

**3.1.6 Pertumbuhan Omset 2024→2025**

In [10]:
fig = px.histogram(master, x="revenue_growth_pct", nbins=40,
                    title="Distribusi Pertumbuhan Omset 2024\u21922025",
                    labels={"revenue_growth_pct": "Pertumbuhan Omset (%)"})
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()


**Insight:** Distribusi mendekati normal (lonceng), rata-rata tumbuh
+11,6%, dengan sebagian kecil mengalami penurunan tajam. Ini fitur numerik
paling "sehat" secara statistik dibanding DSR atau collateral ratio —
variasinya baik untuk membedakan nasabah satu sama lain.

**3.1.7 Sektor Industri**

In [11]:
vc = master["industry"].value_counts().reset_index()
vc.columns = ["industry", "jumlah"]

fig = px.bar(vc, x="industry", y="jumlah",
             title="Distribusi Sektor Industri",
             labels={"industry": "Sektor Industri", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** Merata di keenam sektor (472–523 nasabah masing-masing) —
sesuai desain generator (dipilih random uniform), jadi tidak ada insight
bisnis khusus di sini selain konfirmasi diversifikasi portofolio yang baik.

**3.1.8 Total Current Balance (fitur baru)**

In [12]:
fig = px.histogram(master, x="bank_current_balance_total", nbins=50,
                    title="Distribusi Total Current Balance (semua rekening per nasabah)",
                    labels={"bank_current_balance_total": "Total Current Balance (IDR)"})
fig.show()


**Insight:** `current_balance` (saldo real-time, dijumlah dari semua
rekening 1 nasabah) rata-rata sekitar Rp 28,5 juta, dengan sebaran yang
lebar (std ~Rp 33,2 juta) — mayoritas nasabah punya saldo mengendap yang
cukup sehat relatif terhadap skala pinjaman UMKM. Ada nilai minimum negatif
kecil, hasil dari simulasi kondisi "baru kena overdraft" yang sengaja
dimasukkan ke generator.

**3.1.9 Level RM yang Menangani (fitur baru)**

In [13]:
vc = master["level"].value_counts().reset_index()
vc.columns = ["level", "jumlah"]

fig = px.bar(vc, x="level", y="jumlah", color="level",
             title="Distribusi Nasabah berdasarkan Level RM yang Menangani",
             labels={"level": "Level RM", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** 1.793 nasabah (59,8%) ditangani Junior RB, 1.207 (40,2%)
oleh Senior RB — sesuai proporsi 60:40 yang di-set di generator. Ini murni
komposisi portofolio, bukan indikasi kualitas.

**3.1.10 Usia Pemilik Usaha**

In [14]:
fig = px.histogram(master, x="owner_age", nbins=30,
                    title="Distribusi Usia Pemilik Usaha",
                    labels={"owner_age": "Usia Pemilik (tahun)"})
fig.show()


**Insight:** Terkonsentrasi di rentang 21–60 tahun, drop tajam di atas
60 — masuk akal untuk populasi pemilik UMKM yang masih aktif berusaha.

### 3.2 Bivariate — Hubungan Tiap Fitur dengan Label

**3.2.1 Kolektibilitas SLIK vs Label**

In [15]:
collect_label_map = {0: "Belum Ada Riwayat", 1: "Lancar", 2: "DPK",
                      3: "Kurang Lancar", 4: "Diragukan", 5: "Macet"}
order = ["Belum Ada Riwayat","Lancar","DPK","Kurang Lancar","Diragukan","Macet"]
master["collectability_readable"] = master["slik_worst_collectability"].map(collect_label_map)

ct = pd.crosstab(master["collectability_readable"], master["label"], normalize="index") * 100
ct = ct.reindex(order).reset_index().melt(id_vars="collectability_readable", var_name="label", value_name="persen")

fig = px.bar(ct, x="collectability_readable", y="persen", color="label", barmode="stack",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Kolektibilitas SLIK vs Label (%)",
             labels={"collectability_readable": "Kolektibilitas SLIK Terburuk", "persen": "Persentase (%)"})
fig.show()


**Insight:** Pola jelas dan monoton — makin buruk kolektibilitas
(Lancar → DPK → ... → Macet), makin tinggi proporsi Ditolak. Ini sinyal
paling kuat & paling "bersih" untuk membedakan Diterima/Ditolak, sesuai
bobot Character yang memang paling besar (35%) dalam skema penilaian.

**3.2.2 Status DHN vs Label**

In [16]:
ct = master.groupby(["status_dhn", "label"]).size().reset_index(name="jumlah")

fig = px.bar(ct, x="status_dhn", y="jumlah", color="label", barmode="group",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Status DHN vs Label",
             labels={"status_dhn": "Terdaftar DHN?", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** Nasabah dengan status DHN "Ya" hampir seluruhnya berlabel
Ditolak — konsisten dengan logika `compute_label_score` yang secara
eksplisit membatasi skor Character maksimal 0,1 kalau status DHN aktif.

**3.2.3 Estimasi DSR vs Label**

In [17]:
fig = px.box(master, x="label", y="estimated_dsr", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Estimasi DSR vs Label",
             labels={"estimated_dsr": "Estimasi DSR", "label": "Label"})
fig.show()


**Insight:** Sesuai temuan di univariate, DSR kurang bisa membedakan
kedua kelompok label secara visual (median kedua kelompok sama-sama dekat
ke batas cap) — mendukung catatan sebelumnya bahwa fitur ini butuh
perbaikan representasi untuk dipakai lebih efektif di modeling.

**3.2.4 Collateral Ratio vs Label**

In [18]:
fig = px.box(master, x="label", y="collateral_ratio", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             log_y=True,
             title="Collateral Ratio vs Label (log scale)",
             labels={"collateral_ratio": "Collateral Ratio", "label": "Label"})
fig.show()


**Insight:** Nasabah Ditolak cenderung punya collateral ratio yang
sedikit lebih rendah/lebih variatif di ujung bawah, tapi overlap-nya besar
dengan kelompok Diterima — collateral ratio saja bukan pembeda kuat
(realistis, karena Collateral cuma 20% bobot dalam skema penilaian,
kalah dominan dari Character).

**3.2.5 Sektor Industri vs Label**

In [19]:
ct = master.groupby(["industry", "label"]).size().reset_index(name="jumlah")

fig = px.bar(ct, x="industry", y="jumlah", color="label", barmode="group",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             title="Sektor Industri vs Label",
             labels={"industry": "Sektor Industri", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** Proporsi Diterima/Ditolak relatif mirip di semua sektor —
sesuai desain (`INDUSTRY_RISK` cuma berkontribusi 15% bobot Condition),
jadi sektor usaha bukan penentu dominan kelayakan di skema ini.

**3.2.6 Pertumbuhan Omset vs Label**

In [20]:
fig = px.histogram(master, x="revenue_growth_pct", color="label", barmode="overlay",
                    nbins=40, opacity=0.65,
                    color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
                    title="Pertumbuhan Omset vs Label",
                    labels={"revenue_growth_pct": "Pertumbuhan Omset (%)"})
fig.show()


**Insight:** Kelompok Ditolak jelas condong ke pertumbuhan omset
negatif/rendah dibanding kelompok Diterima — pemisahan yang cukup terlihat
lewat overlay histogram ini, sejalan dengan bobot Capacity yang besar
(30%).

**3.2.7 Total Current Balance vs Label**

In [21]:
fig = px.box(master, x="label", y="bank_current_balance_total", color="label",
             color_discrete_map={"Diterima": "#2ecc71", "Ditolak": "#e74c3c"},
             log_y=True,
             title="Total Current Balance vs Label (log scale)",
             labels={"bank_current_balance_total": "Total Current Balance (IDR)", "label": "Label"})
fig.show()


**Insight:** Tidak ada perbedaan besar antara kedua kelompok label —
masuk akal karena `current_balance` **tidak dipakai sama sekali** dalam
`compute_label_score` (label dihitung sebelum kolom ini ada). Ini justru
bagus sebagai bahan diskusi: `current_balance` berpotensi jadi fitur
**baru & independen** untuk penilaian Cashflow ke depannya, karena
belum "ikut campur" menentukan label historis (sehingga tidak redundan).

**3.2.8 Approval Rate per Cabang (lewat RM) — fitur baru**

In [22]:
rm_summary = master.groupby(["rm_branch_name"]).agg(
    jumlah_nasabah=("NIK", "count"),
    approval_rate=("label", lambda s: (s == "Diterima").mean() * 100)
).reset_index().sort_values("approval_rate")

fig = px.bar(rm_summary, x="rm_branch_name", y="approval_rate",
             title="Approval Rate per Cabang (berdasarkan RM yang menangani)",
             labels={"rm_branch_name": "Cabang RM", "approval_rate": "Approval Rate (%)"})
fig.add_hline(y=(master["label"]=="Diterima").mean()*100, line_dash="dash", line_color="gray",
              annotation_text="rata-rata keseluruhan")
fig.show()


**Insight:** Ada variasi approval rate antar cabang, tapi semuanya
berkisar di sekitar rata-rata keseluruhan (garis putus-putus) tanpa
outlier ekstrem — wajar, karena `branch_name`/`rm_id` juga tidak dipakai
dalam `compute_label_score`. Kalau nanti ditemukan cabang yang jauh
menyimpang, itu justru sinyal bagus untuk digali lebih lanjut di halaman
Monitoring (apakah portofolio nasabahnya memang lebih berisiko, atau ada
faktor lain).

### 3.3 Korelasi Antar Fitur Numerik

In [23]:
numeric_cols = ["owner_age","business_age_year","employee_count","monthly_turnover_est",
                "loan_requested","collateral_ratio","collateral_size_m2","estimated_dsr",
                "slik_worst_collectability","slik_n_loans","revenue_growth_pct",
                "profit_margin_2025","liability_to_asset_2025","bank_best_avg_balance_6m",
                "bank_current_balance_total","bank_total_overdraft_6m"]
corr = master[numeric_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu_r",
                 zmin=-1, zmax=1, title="Korelasi Antar Fitur Numerik Utama")
fig.update_layout(height=700)
fig.show()


**Insight — korelasi terkuat yang ditemukan:**

| Pasangan fitur | Korelasi | Penjelasan |
|---|---|---|
| `bank_best_avg_balance_6m` vs `bank_current_balance_total` | **0,82** | Wajar — keduanya sama-sama diturunkan dari omset bulanan nasabah di generator |
| `monthly_turnover_est` vs `bank_best_avg_balance_6m` | 0,72 | Saldo rekening memang dirancang proporsional terhadap omset |
| `bank_current_balance_total` vs `monthly_turnover_est` | 0,59 | Sama alasannya |
| `slik_n_loans` vs `slik_worst_collectability` | 0,49 | Makin banyak fasilitas kredit yang dipegang, makin besar peluang salah satunya bermasalah |
| `collateral_ratio` vs `loan_requested` | **-0,47** | Masuk akal — makin besar pinjaman yang diajukan, makin kecil rasio agunan/pinjaman-nya (agunan tidak ikut membesar proporsional) |
| `monthly_turnover_est` vs `estimated_dsr` | -0,39 | Omset lebih besar → DSR (rasio cicilan) cenderung lebih kecil |

**Implikasi untuk modeling:** `bank_best_avg_balance_6m` dan
`bank_current_balance_total` berkorelasi tinggi (0,82) — kalau keduanya
dipakai sebagai fitur model bersamaan, pertimbangkan multicollinearity
(bisa pilih salah satu, atau gabungkan jadi 1 fitur turunan). Tidak
ditemukan fitur numerik yang korelasinya sangat tinggi (>0,9) satu sama
lain di luar pasangan yang secara desain memang terkait, jadi risiko
redundansi fitur secara umum masih terkendali.

## 4. Agentic Screening Pipeline

Implementasi 7 sub-agent (rule-based), sesuai arsitektur yang sudah
disepakati sebelumnya:

| Agent | Tugas | Fitur yang dipakai |
|---|---|---|
| Identity Agent | Validasi Dukcapil | `NIK`, `owner_age` |
| Credit History Agent | Cek SLIK | `slik_worst_collectability`, `slik_n_banks`, `slik_has_macet` |
| DHN Agent | Cek daftar hitam | `status_dhn` |
| Collateral Agent | Validasi ATR/BPN | `collateral_ratio`, `ownership_match` |
| Financial Agent | Analisis laporan keuangan | `revenue_growth_pct`, `profit_margin_2025` |
| Cashflow Agent | Analisis mutasi rekening | `bank_best_avg_balance_6m`, **`bank_best_current_balance` (baru)**, `bank_total_overdraft_6m`, `bank_any_dormant` |
| **Risk Agent** | Orkestrator — gabungkan semua + hard rules + keputusan & narasi | Semua skor di atas + `industry`, `loan_requested`, `collateral_market_value` |

### ⚠️ Penyesuaian penting dari versi sebelumnya

**Cashflow Agent dikalibrasi ulang.** Karena generator `bank_account`
sekarang pakai multiplier saldo 40–100x (dulu 2–6x), rasio saldo-terhadap-omset
jadi jauh lebih besar (median ~10x omset bulanan, dulu mendekati 1x). Kalau
rumus lama (`clip(rasio, 0, 1)`) dipakai apa adanya, hampir semua nasabah
akan mentok skor 1.0 — tidak diskriminatif sama sekali. Skor sekarang
dinormalisasi pakai **persentil ke-90 dari data historis** sebagai acuan
"skor penuh" (bukan angka sembarang), dan **`current_balance`** (saldo
real-time, fitur baru) ikut digabung 50:50 dengan `average_balance_6m`
(saldo rata-rata 6 bulan) sebagai sinyal independen.

**Narasi insight sekarang mengimplementasikan lengkap 6 kategori** sesuai
spesifikasi awal (sebelumnya versi awal cuma pakai 4 dari 6): *layak
karena*, **layak tapi** (baru — dulu ini bug: kategori "Layak" tidak
pernah mengecek skor komponen individual, jadi bisa salah klaim "semua
aman" padahal ada yang lemah), *layak bersyarat karena*, *perlu review
ulang karena*, **tidak layak tapi** (baru), *tidak layak karena*.

In [24]:
# =========================================================================
# AGENT 1: IDENTITY AGENT (validasi Dukcapil) - TIDAK BERUBAH dari sebelumnya
# =========================================================================
def identity_agent(row):
    valid_nik = len(str(row["NIK"])) == 16
    valid_age = row["owner_age"] >= 21
    passed = valid_nik and valid_age
    notes = []
    if not valid_nik: notes.append("Format NIK tidak valid")
    if not valid_age: notes.append("Usia pemohon di bawah 21 tahun")
    return pd.Series({
        "identity_passed": passed,
        "identity_notes": "; ".join(notes) if notes else "Identitas valid",
    })

# =========================================================================
# AGENT 2: CREDIT HISTORY AGENT (SLIK -> Character) - TIDAK BERUBAH
# =========================================================================
def credit_history_agent(row):
    if row["slik_has_credit_history"] == 0:
        score = 0.6
        notes = "Belum memiliki riwayat kredit di SLIK (nasabah baru)"
    else:
        collect = row["slik_worst_collectability"]
        score = {1: 1.0, 2: 0.75, 3: 0.45, 4: 0.2, 5: 0.0}.get(int(collect), 0.5)
        label_map = {1:"Lancar",2:"Dalam Perhatian Khusus",3:"Kurang Lancar",4:"Diragukan",5:"Macet"}
        notes = f"Kolektibilitas terburuk: {label_map.get(int(collect))} di {int(row['slik_n_banks'])} bank"
        if row["slik_has_macet"] == 1:
            notes += " — riwayat Macet ditemukan"
    return pd.Series({"character_score": round(score, 3), "character_notes": notes})

# =========================================================================
# AGENT 3: DHN AGENT - TIDAK BERUBAH
# =========================================================================
def dhn_agent(row):
    blacklisted = row["status_dhn"] == "Ya"
    notes = row["dhn_alasan"] if blacklisted else "Tidak terdaftar di Daftar Hitam Nasional"
    return pd.Series({"dhn_blacklisted": blacklisted, "dhn_notes": notes})

# =========================================================================
# AGENT 4: COLLATERAL AGENT - TIDAK BERUBAH (skala collateral_ratio tetap sama)
# =========================================================================
def collateral_agent(row):
    ratio = row["collateral_ratio"]
    match = row["ownership_match"] == "Ya"
    score = np.clip(ratio / 1.5, 0, 1)
    if not match:
        score = min(score, 0.2)
    notes = f"LTV agunan {ratio*100:.0f}% dari pinjaman"
    if not match:
        notes += " — nama sertifikat TIDAK sesuai pemilik (butuh verifikasi manual)"
    return pd.Series({"collateral_score": round(float(score), 3), "collateral_notes": notes})

# =========================================================================
# AGENT 5: FINANCIAL AGENT - TIDAK BERUBAH (skala revenue_growth/margin tetap sama)
# =========================================================================
def financial_agent(row):
    growth = row["revenue_growth_pct"]
    margin = row["profit_margin_2025"]
    score = np.clip(0.5 + growth, 0, 1) * 0.6 + np.clip(margin / 0.15, 0, 1) * 0.4
    trend = "tumbuh" if growth > 0.02 else ("stagnan" if growth > -0.02 else "menurun")
    notes = f"Omset {trend} {growth*100:+.1f}% (2024→2025), margin laba {margin*100:.1f}%"
    return pd.Series({"financial_score": round(float(np.clip(score,0,1)), 3), "financial_notes": notes})

# =========================================================================
# AGENT 6: CASHFLOW AGENT - DIKALIBRASI ULANG untuk skala data terbaru
# =========================================================================
# average_balance_6m & current_balance sekarang jauh lebih besar relatif thd
# omset bulanan (multiplier generator baru 40-100x, dulu 2-6x) - kalau rumus
# lama (rasio langsung di-clip 0..1) dipakai apa adanya, hampir semua nasabah
# akan mentok skor 1.0 (tidak diskriminatif). Dikalibrasi ulang pakai
# persentil ke-90 dari data historis sebagai acuan "skor penuh", BUKAN angka
# sembarang - supaya sebarannya tetap informatif dgn skala data saat ini.
BALANCE_RATIO_P90 = 19.2   # persentil-90 bank_best_avg_balance_6m / monthly_turnover_est
CURRENT_RATIO_P90 = 22.8   # persentil-90 bank_best_current_balance / monthly_turnover_est

def cashflow_agent(row):
    monthly_turnover = max(row["monthly_turnover_est"], 1)
    balance_ratio = row["bank_best_avg_balance_6m"] / monthly_turnover
    current_ratio = row["bank_best_current_balance"] / monthly_turnover

    balance_score = np.clip(balance_ratio / BALANCE_RATIO_P90, 0, 1)
    current_score = np.clip(current_ratio / CURRENT_RATIO_P90, 0, 1)
    base_score = 0.5 * balance_score + 0.5 * current_score

    overdraft_penalty = min(row["bank_total_overdraft_6m"] * 0.1, 0.3)
    dormant_penalty = 0.15 if row["bank_any_dormant"] == 1 else 0
    score = np.clip(base_score - overdraft_penalty - dormant_penalty, 0, 1)

    notes = (f"Saldo rata-rata {balance_ratio:.1f}x omset bulanan, "
             f"saldo real-time {current_ratio:.1f}x omset bulanan")
    if row["bank_total_overdraft_6m"] > 0:
        notes += f", overdraft {int(row['bank_total_overdraft_6m'])}x dalam 6 bulan"
    if row["bank_any_dormant"] == 1:
        notes += ", memiliki rekening dormant"
    return pd.Series({"cashflow_score": round(float(score), 3), "cashflow_notes": notes})

# =========================================================================
# AGENT 7: RISK AGENT (orkestrator)
# =========================================================================
INDUSTRY_RISK_PENALTY = {"Perdagangan":0.02,"Kuliner":0.04,"Jasa":0.02,
                          "Manufaktur":0.03,"Pertanian":0.06,"Transportasi":0.05}
INTEREST_BY_ZONE = {"Hijau": 9.5, "Kuning": 12.0, "Merah": 15.0}
TENOR_BY_LOAN = {"KMK": 12, "KI": 36, "KPR": 120, "KKB": 48, "KK": 24}
STRONG, WEAK = 0.7, 0.5

def _compose_insight(decision, scores, score, hard_rule_reason=None):
    """scores: dict {nama_agent: skor}. Implementasi lengkap 6 kategori narasi
    sesuai spesifikasi: layak karena / layak tapi / tidak layak karena /
    tidak layak tapi / layak bersyarat karena / perlu review ulang karena."""
    weak = [k for k, v in scores.items() if v < WEAK]
    strong = [k for k, v in scores.items() if v >= STRONG]

    if hard_rule_reason:
        return f"Tidak layak karena {hard_rule_reason}."

    if decision == "Layak":
        if weak:
            return (f"Layak tapi {', '.join(weak)} tergolong lemah (skor di bawah 0.5) "
                     "— disarankan tetap dimonitor meski keputusan akhir disetujui.")
        return (f"Layak karena seluruh komponen (" +
                ", ".join(f"{k} {v:.2f}" for k, v in scores.items()) +
                ") berada di zona aman.")

    if decision == "Layak Bersyarat":
        alasan = weak if weak else ["beberapa indikator berada di batas ambang"]
        return f"Layak bersyarat karena {', '.join(alasan)} — disarankan tambahan agunan/penjamin atau plafon diturunkan."

    if decision == "Perlu Review Ulang":
        return (f"Perlu review ulang karena skor gabungan ({score:.2f}) berada di area abu-abu "
                 "— disarankan OTS/wawancara lanjutan sebelum keputusan final.")

    # Tidak Layak (dari threshold skor, bukan hard rule)
    if strong:
        return (f"Tidak layak tapi {', '.join(strong)} tergolong kuat — skor gabungan "
                 f"({score:.2f}) masih di bawah ambang, bisa dipertimbangkan ulang jika "
                 "ada mitigasi risiko dari sisi lain.")
    return f"Tidak layak karena skor gabungan ({score:.2f}) di bawah ambang batas kelayakan pada hampir seluruh komponen."


def risk_agent(row):
    # --- Hard rules (kill-switch) ---
    if row["identity_passed"] == False:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None, "risk_score": None,
            "insight": _compose_insight("Tidak Layak", {}, 0, row["identity_notes"].lower()),
        })
    if row["dhn_blacklisted"]:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None, "risk_score": None,
            "insight": _compose_insight("Tidak Layak", {}, 0,
                f"nasabah terdaftar di Daftar Hitam Nasional ({row['dhn_notes']})"),
        })
    if row["character_score"] == 0.0:
        return pd.Series({
            "decision": "Tidak Layak", "zone": "Merah",
            "jenis_kredit_rekomendasi": "-", "nominal_disetujui": 0,
            "jangka_waktu_bulan": 0, "bunga_persen": None, "risk_score": None,
            "insight": _compose_insight("Tidak Layak", {}, 0, "memiliki riwayat kredit Macet pada SLIK"),
        })

    # --- Weighted score ---
    industry_penalty = INDUSTRY_RISK_PENALTY.get(row["industry"], 0.03)
    condition_score = np.clip(1 - industry_penalty * 4, 0, 1)
    score = (0.35 * row["character_score"] + 0.25 * row["financial_score"]
             + 0.20 * row["collateral_score"] + 0.10 * row["cashflow_score"]
             + 0.10 * condition_score)

    if score >= 0.70:
        decision, zone = "Layak", "Hijau"
    elif score >= 0.55:
        decision, zone = "Layak Bersyarat", "Kuning"
    elif score >= 0.40:
        decision, zone = "Perlu Review Ulang", "Kuning"
    else:
        decision, zone = "Tidak Layak", "Merah"

    max_by_collateral = row["collateral_market_value"] * 0.7
    nominal = int(min(row["loan_requested"], max_by_collateral)) if decision != "Tidak Layak" else 0
    jenis = "KMK" if row["loan_requested"] < 200_000_000 else "KI"
    tenor = TENOR_BY_LOAN.get(jenis, 24) if decision != "Tidak Layak" else 0
    bunga = INTEREST_BY_ZONE[zone] if decision != "Tidak Layak" else None

    scores = {"Character": row["character_score"], "Financial": row["financial_score"],
              "Collateral": row["collateral_score"], "Cashflow": row["cashflow_score"]}
    insight = _compose_insight(decision, scores, score)

    return pd.Series({
        "decision": decision, "zone": zone,
        "jenis_kredit_rekomendasi": jenis, "nominal_disetujui": nominal,
        "jangka_waktu_bulan": tenor, "bunga_persen": bunga,
        "risk_score": round(float(score), 3), "insight": insight,
    })

In [25]:
print("Menjalankan Identity Agent...")
master = master.join(master.apply(identity_agent, axis=1))
print("Menjalankan Credit History Agent...")
master = master.join(master.apply(credit_history_agent, axis=1))
print("Menjalankan DHN Agent...")
master = master.join(master.apply(dhn_agent, axis=1))
print("Menjalankan Collateral Agent...")
master = master.join(master.apply(collateral_agent, axis=1))
print("Menjalankan Financial Agent...")
master = master.join(master.apply(financial_agent, axis=1))
print("Menjalankan Cashflow Agent...")
master = master.join(master.apply(cashflow_agent, axis=1))
print("Menjalankan Risk Agent (orkestrator)...")
master = master.join(master.apply(risk_agent, axis=1))

print("\nDistribusi keputusan Agentic Pipeline:")
print(master["decision"].value_counts(normalize=True).round(3))

master[["application_id","company_name","decision","zone","jenis_kredit_rekomendasi",
        "nominal_disetujui","jangka_waktu_bulan","bunga_persen","insight"]].head(5)


Menjalankan Identity Agent...
Menjalankan Credit History Agent...
Menjalankan DHN Agent...
Menjalankan Collateral Agent...
Menjalankan Financial Agent...
Menjalankan Cashflow Agent...
Menjalankan Risk Agent (orkestrator)...

Distribusi keputusan Agentic Pipeline:
decision
Layak                 0.688
Layak Bersyarat       0.206
Tidak Layak           0.075
Perlu Review Ulang    0.031
Name: proportion, dtype: float64


,application_id,company_name,decision,zone,jenis_kredit_rekomendasi,nominal_disetujui,jangka_waktu_bulan,bunga_persen,insight
0,APP202600001,UD Santoso Abadi,Layak,Hijau,KI,300000000,36,9.5,Layak tapi Cashflow tergolong lemah (skor di b...
1,APP202600002,UD Wijaya Mandiri,Layak,Hijau,KMK,75000000,12,9.5,Layak tapi Cashflow tergolong lemah (skor di b...
2,APP202600003,UD Kusuma Sejahtera,Perlu Review Ulang,Kuning,KMK,150000000,12,12.0,Perlu review ulang karena skor gabungan (0.50)...
3,APP202600004,UD Susanto Makmur,Layak,Hijau,KI,200000000,36,9.5,Layak tapi Cashflow tergolong lemah (skor di b...
4,APP202600005,CV Wijaya Sejahtera,Layak,Hijau,KI,750000000,36,9.5,"Layak karena seluruh komponen (Character 0.75,..."


### 4.1 Distribusi 6 Kategori Narasi Insight

In [26]:
def kategori_insight(s):
    for prefix in ["Layak bersyarat karena", "Perlu review ulang karena", "Layak tapi",
                   "Layak karena", "Tidak layak tapi", "Tidak layak karena"]:
        if s.lower().startswith(prefix.lower()):
            return prefix
    return "Lainnya"

master["insight_kategori"] = master["insight"].apply(kategori_insight)
vc = master["insight_kategori"].value_counts().reset_index()
vc.columns = ["kategori", "jumlah"]

order = ["Layak karena", "Layak tapi", "Layak bersyarat karena",
         "Perlu review ulang karena", "Tidak layak tapi", "Tidak layak karena"]
vc["kategori"] = pd.Categorical(vc["kategori"], categories=order, ordered=True)
vc = vc.sort_values("kategori")

fig = px.bar(vc, x="kategori", y="jumlah",
             title="Distribusi 6 Kategori Narasi Insight",
             labels={"kategori": "Kategori Insight", "jumlah": "Jumlah Nasabah"})
fig.show()


**Insight:** Kategori **"Layak tapi"** justru paling banyak (lebih dari
separuh kelompok Layak) — artinya banyak nasabah yang lolos keputusan akhir
tapi punya minimal 1 komponen skor di bawah 0,5 (paling sering Cashflow,
karena baru dikalibrasi ulang jadi lebih tersebar/ketat). Ini nunjukkin
manfaat langsung dari perbaikan narasi: keputusan akhir tetap Layak, tapi
sistem tetap "jujur" menandai titik lemahnya — berguna buat RB yang mau
follow-up meski pengajuan sudah disetujui.

Kategori **"Tidak layak tapi"** kosong (0 nasabah) pada data ini — bukan
bug, murni karakteristik statistik: begitu skor gabungan jatuh di bawah
0,40, secara matematis kecil kemungkinan ada 1 komponen yang tetap sangat
kuat (≥0,7) sampai "menyelamatkan" narasinya. Kategori ini tetap valid
secara struktur, hanya belum muncul di run data ini.

### 4.2 Validasi Pipeline vs Label Asli

In [27]:
cm = pd.crosstab(master["decision"], master["label"])
cm = cm.reindex(["Layak", "Layak Bersyarat", "Perlu Review Ulang", "Tidak Layak"])

fig = px.imshow(cm, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                 title="Keputusan Agentic Pipeline vs Label Asli (Ground Truth)",
                 labels={"x": "Label Asli (dari generator)", "y": "Keputusan Agent Pipeline",
                         "color": "Jumlah"})
fig.show()

agree_rate = (master["decision"].isin(["Layak","Layak Bersyarat"]) == (master["label"]=="Diterima")).mean()
print(f"Tingkat kesesuaian keputusan agent pipeline vs label asli: {agree_rate:.1%}")
print()
print("Distribusi zona risiko:")
print(master["zone"].value_counts(normalize=True).round(3))


Tingkat kesesuaian keputusan agent pipeline vs label asli: 92.7%

Distribusi zona risiko:
zone
Hijau     0.688
Kuning    0.237
Merah     0.075
Name: proportion, dtype: float64


**Insight:** Tingkat kesesuaian keputusan pipeline vs label asli ada di
output di atas — sanity check ini bukan evaluasi model ML (pipeline-nya
rule-based, bukan trained model), tapi indikasi bahwa logika 7-agent cukup
dekat dengan logika yang dipakai generator untuk membuat label ground
truth, meski beda persis (Risk Agent punya hard rules & bobot Financial/Cashflow
yang berbeda dari `compute_label_score` generator).

## 5. Export `master_scored.csv`

In [28]:
export_cols = ["application_id","NIK","company_name","owner_name","industry",
    "sub_industry","province","city","branch_name","region",
    "rm_id","rm_name","rm_branch_name","level",   # kolom RM: monitoring saja, JANGAN dipakai sbg fitur model
    "loan_requested","collateral_type","collateral_market_value","collateral_ratio",
    "estimated_dsr","revenue_growth_pct","profit_margin_2025",
    "slik_worst_collectability","status_dhn",
    "bank_best_avg_balance_6m","bank_best_current_balance",
    "identity_passed","character_score","character_notes",
    "collateral_score","collateral_notes","financial_score","financial_notes",
    "cashflow_score","cashflow_notes","risk_score",
    "decision","zone","jenis_kredit_rekomendasi","nominal_disetujui",
    "jangka_waktu_bulan","bunga_persen","insight","insight_kategori","label"]

master_export = master[export_cols].copy()

PROCESSED_DIR = "../data/processed"  # kalau belum ada di notebook

out_path = f"{PROCESSED_DIR}/master_scored.csv"
master_export.to_csv(out_path, index=False)

print(f"Tersimpan: {out_path}  ({master_export.shape[0]} baris x {master_export.shape[1]} kolom)")
master_export.head(3)


Tersimpan: ../data/processed/master_scored.csv  (3000 baris x 44 kolom)


,application_id,NIK,company_name,owner_name,industry,sub_industry,province,city,branch_name,region,rm_id,rm_name,rm_branch_name,level,loan_requested,collateral_type,collateral_market_value,collateral_ratio,estimated_dsr,revenue_growth_pct,profit_margin_2025,slik_worst_collectability,status_dhn,bank_best_avg_balance_6m,bank_best_current_balance,identity_passed,character_score,character_notes,collateral_score,collateral_notes,financial_score,financial_notes,cashflow_score,cashflow_notes,risk_score,decision,zone,jenis_kredit_rekomendasi,nominal_disetujui,jangka_waktu_bulan,bunga_persen,insight,insight_kategori,label
0,APP202600001,3276010601750001,UD Santoso Abadi,Budi Panjaitan,Manufaktur,Konveksi,Jawa Barat,Depok,KCP Bogor Baranangsiang,Region 2,RM0020,Indah Kusuma,KCP Bogor Baranangsiang,Junior RB,300000000,Rumah,2328496000,7.76,3.0,0.0809,0.1035,1.0,Tidak,19800614,13581790,True,1.00,Kolektibilitas terburuk: Lancar di 1 bank,1.0,LTV agunan 776% dari pinjaman,0.625,"Omset tumbuh +8.1% (2024→2025), margin laba 10.3%",0.304,"Saldo rata-rata 7.4x omset bulanan, saldo real...",0.825,Layak,Hijau,KI,300000000,36,9.5,Layak tapi Cashflow tergolong lemah (skor di b...,Layak tapi,Diterima
1,APP202600002,3172010301920002,UD Wijaya Mandiri,Andi Hidayat,Jasa,Bengkel,DKI Jakarta,Jakarta Utara,KCP Bekasi Barat,Region 1,RM0010,Indah Rahman,KCP Bekasi Barat,Senior RB,75000000,Rumah,3724038000,49.65,3.0,-0.0292,0.1199,2.0,Tidak,4014256,1718267,True,0.75,Kolektibilitas terburuk: Dalam Perhatian Khusu...,1.0,LTV agunan 4965% dari pinjaman,0.602,"Omset menurun -2.9% (2024→2025), margin laba 1...",0.079,"Saldo rata-rata 2.2x omset bulanan, saldo real...",0.713,Layak,Hijau,KMK,75000000,12,9.5,Layak tapi Cashflow tergolong lemah (skor di b...,Layak tapi,Diterima
2,APP202600003,3671010604800003,UD Kusuma Sejahtera,Doni Pratama,Transportasi,Ekspedisi Kecil,Banten,Tangerang,KCP Cibubur,Region 3,RM0039,Slamet Hutapea,KCP Cibubur,Senior RB,150000000,Tanah,1681700000,11.21,3.0,-0.1476,0.1801,4.0,Tidak,14489012,3156322,True,0.20,Kolektibilitas terburuk: Diragukan di 2 bank,1.0,LTV agunan 1121% dari pinjaman,0.611,"Omset menurun -14.8% (2024→2025), margin laba ...",0.013,"Saldo rata-rata 8.5x omset bulanan, saldo real...",0.504,Perlu Review Ulang,Kuning,KMK,150000000,12,12.0,Perlu review ulang karena skor gabungan (0.50)...,Perlu review ulang karena,Ditolak


**Catatan kolom RM di file export:** `rm_id`, `rm_name`, `rm_branch_name`,
`level` disertakan untuk keperluan **monitoring/dashboard** (mis. halaman
"Kinerja RM"), TAPI tidak pernah dipakai sebagai input ke 7 agent atau
skema skor manapun di atas — konsisten dengan keputusan governance yang
sudah dibahas sebelumnya (mencegah bias/favoritism RM individual terkunci
ke sistem screening).